# Смеси гауссиан и алгоритм EM

> Модуль **Unsupervised Learning**, ноутбук 3. Справочный, академичный. Здесь мы переходим от
> жёсткой кластеризации (K-Means) к **мягкой, вероятностной**: каждый объект принадлежит
> кластерам *с вероятностями*, а сами кластеры могут быть вытянутыми эллипсами, а не шарами.

### План занятия
1. **Зачем нужна мягкая кластеризация** — ограничения K-Means.
2. **Вероятностная модель** — гауссиана и смесь гауссиан (Gaussian Mixture Model, GMM).
3. **Ответственности** — мягкие (вероятностные) назначения.
4. **Алгоритм EM** — E-шаг и M-шаг, формулы пересчёта, сходимость, вывод.
5. **Связь с K-Means** — почему K-Means это частный (предельный) случай GMM.
6. **Типы ковариаций** — `spherical / diag / tied / full` и какие формы они допускают.
7. **Выбор числа компонент** — по BIC/AIC.
8. **Практика** — дикторы по эмбеддингам, на этот раз с оценкой уверенности.

В конце — раздел **вопросов и ответов**.

## 1. Зачем нужна мягкая кластеризация

У K-Means два жёстких ограничения (вспомним прошлый ноутбук):

- **Жёсткие метки.** Объект целиком принадлежит ровно одному кластеру, даже если он лежит
  ровно на границе между двумя. Нет понятия «на 60% диктор A, на 40% диктор B».
- **Сферические кластеры.** Критерий $\sum\|x-\mu\|^2$ меряет расстояние одинаково во все
  стороны, поэтому K-Means видит только круглые «шары» одинакового размера.

**Gaussian Mixture Model (GMM)** снимает оба ограничения. Он смотрит на данные как на смесь
нескольких **гауссовых «облаков»**, каждое из которых может быть вытянуто и повёрнуто, и для
каждого объекта выдаёт **вероятности** принадлежности к каждому облаку. Учат такую модель
алгоритмом **EM (Expectation–Maximization)**.

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Ellipse

from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

try:
    from ipywidgets import interact, IntSlider
    HAVE_WIDGETS = True
except Exception:
    HAVE_WIDGETS = False

np.random.seed(0)
plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": True, "grid.alpha": 0.3})
PALETTE = ["#2f6f4f", "#2d5d8a", "#9b3b46", "#9a6b00", "#5b4b8a", "#3a7d7b"]

# Three anisotropic (elliptical) gaussian clouds — exactly what GMM is built for
rng = np.random.default_rng(1)
X_gmm = np.vstack([
    rng.multivariate_normal([0, 0],  [[3.0, 1.5], [1.5, 1.0]], 220),
    rng.multivariate_normal([7, 4],  [[1.0, -0.9], [-0.9, 1.6]], 220),
    rng.multivariate_normal([0, 6],  [[2.2, 0.0], [0.0, 0.35]], 220),
])
print("data:", X_gmm.shape)

## 2. Вероятностная модель: смесь гауссиан

### Одна гауссиана

Многомерное нормальное (гауссово) распределение задаётся **центром** $\mu$ и **матрицей
ковариаций** $\Sigma$ (форма и ориентация облака):

$$ \mathcal{N}(x \mid \mu, \Sigma) =
   \frac{1}{(2\pi)^{d/2}\,|\Sigma|^{1/2}}
   \exp\!\left(-\tfrac{1}{2}(x-\mu)^{\top}\Sigma^{-1}(x-\mu)\right). $$

Матрица $\Sigma$ — это и есть то, что позволяет облаку быть **вытянутым и повёрнутым** (в
отличие от «шара» K-Means).

### Смесь $K$ гауссиан

Предполагаем, что данные порождены так: сначала случайно выбирается одно из $K$ облаков (с
вероятностью $\pi_k$, это **веса смеси**, $\sum_k \pi_k = 1$), затем точка берётся из
соответствующей гауссианы. Плотность смеси:

$$ p(x) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(x \mid \mu_k, \Sigma_k). $$

Параметры модели — это $\{\pi_k, \mu_k, \Sigma_k\}_{k=1}^{K}$. **Кластеризовать** = подобрать
эти параметры так, чтобы смесь как можно лучше описывала данные, и затем для каждого объекта
посмотреть, из какого облака он вероятнее всего.

> **Скрытая переменная.** Какое облако породило точку — мы не наблюдаем; это *скрытая*
> (латентная) переменная $z_n \in \{1, \dots, K\}$. Именно из-за неё обычная максимизация
> правдоподобия становится сложной и приводит к EM.

## 3. Ответственности: мягкие назначения

Если параметры известны, то по формуле Байеса можно посчитать **вероятность того, что объект
$x_n$ порождён облаком $k$** — её называют **ответственностью** (responsibility):

$$ \gamma_{nk} = \frac{\pi_k \, \mathcal{N}(x_n \mid \mu_k, \Sigma_k)}
                      {\sum_{j=1}^{K} \pi_j \, \mathcal{N}(x_n \mid \mu_j, \Sigma_j)}. $$

Это число в $[0,1]$, и $\sum_k \gamma_{nk} = 1$. В отличие от жёсткой метки K-Means, объект на
границе получит, например, $\gamma = (0.55, 0.45)$ — модель честно сообщает о неуверенности.
Жёсткую метку при желании получают как $\arg\max_k \gamma_{nk}$.

## 4. Алгоритм EM (Expectation–Maximization)

Возникает «курица и яйцо»: чтобы посчитать ответственности $\gamma$, нужны параметры
$\{\pi, \mu, \Sigma\}$; чтобы оценить параметры, нужно знать, какие точки какому облаку
принадлежат (то есть $\gamma$). EM разрывает этот круг, чередуя два шага:

**E-шаг (Expectation).** При текущих параметрах посчитать ответственности $\gamma_{nk}$ по
формуле из раздела 3.

**M-шаг (Maximization).** При фиксированных $\gamma$ обновить параметры как
**взвешенные** оценки (вес объекта в кластере = его ответственность). Введём «эффективное
число точек» кластера $N_k = \sum_n \gamma_{nk}$:

$$ \mu_k = \frac{1}{N_k}\sum_n \gamma_{nk}\, x_n, \qquad
   \Sigma_k = \frac{1}{N_k}\sum_n \gamma_{nk}\,(x_n - \mu_k)(x_n - \mu_k)^{\top}, \qquad
   \pi_k = \frac{N_k}{N}. $$

Шаги повторяют до сходимости. Каждая итерация **не уменьшает** правдоподобие данных
$\log p(X) = \sum_n \log \sum_k \pi_k \mathcal{N}(x_n\mid\mu_k,\Sigma_k)$, поэтому EM сходится —
но, как и K-Means, к **локальному** максимуму (поэтому на практике инициализируют через
K-Means и делают несколько запусков).

<details>
<summary><b>Для любознательных:</b> почему EM работает (схема вывода)</summary>

Напрямую максимизировать $\log p(X) = \sum_n \log \sum_k \pi_k \mathcal{N}(x_n\mid\theta_k)$
мешает логарифм суммы. Трюк: для любого распределения $q_n(k)$ по неравенству Йенсена (то
самое, что мы выводили в ноутбуке про деревья: $\log$ вогнут, поэтому $\log\mathbb{E}\ge
\mathbb{E}\log$)

$$ \log\sum_k \pi_k\mathcal{N}(x_n\mid\theta_k)
   = \log\sum_k q_n(k)\frac{\pi_k\mathcal{N}(x_n\mid\theta_k)}{q_n(k)}
   \;\ge\; \sum_k q_n(k)\log\frac{\pi_k\mathcal{N}(x_n\mid\theta_k)}{q_n(k)} =: \mathcal{L}(q,\theta). $$

$\mathcal{L}$ — это **нижняя граница** правдоподобия. EM поднимает её по очереди по двум
аргументам:
- **E-шаг** максимизирует $\mathcal{L}$ по $q$ при фиксированных $\theta$. Оптимум —
  $q_n(k)=\gamma_{nk}$ (апостериорная вероятность); при этом граница касается правдоподобия.
- **M-шаг** максимизирует $\mathcal{L}$ по $\theta$ при фиксированных $q=\gamma$. Приравняв
  производные нулю, получаем именно формулы взвешенных среднего и ковариации выше.

Поскольку каждый шаг не уменьшает $\mathcal{L}$, а E-шаг прижимает границу к правдоподобию,
само правдоподобие монотонно растёт. Заметьте: формула $\mu_k$ — это снова «лучшая константа»
(взвешенное среднее), как в листе дерева и в K-Means.
</details>

Посмотрим EM в действии — слайдер прокручивает итерации. Эллипсы — это облака $\Sigma_k$
(уровень $2\sigma$), цвет точки — **смесь** цветов кластеров по ответственностям (точки на
границе выглядят «промежуточным» цветом):

In [ ]:
# Manual EM so we can watch every iteration (responsibilities + covariance ellipses)
def gaussian_pdf(X, mu, cov):
    d = X.shape[1]
    diff = X - mu
    inv = np.linalg.inv(cov)
    expo = -0.5 * np.einsum("ni,ij,nj->n", diff, inv, diff)
    return np.exp(expo) / np.sqrt((2 * np.pi) ** d * np.linalg.det(cov))


def em_history(X, k, seed, n_iter=25):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    mus = X[rng.choice(n, k, replace=False)].astype(float)
    covs = np.array([np.cov(X.T) for _ in range(k)])     # start broad
    pis = np.full(k, 1.0 / k)
    hist = []
    for _ in range(n_iter):
        # E-step: responsibilities
        r = np.stack([pis[c] * gaussian_pdf(X, mus[c], covs[c]) for c in range(k)], axis=1)
        tot = r.sum(1, keepdims=True)
        loglik = np.log(np.clip(tot, 1e-300, None)).sum()
        r = r / np.clip(tot, 1e-12, None)
        hist.append((pis.copy(), mus.copy(), covs.copy(), r.copy(), loglik))
        # M-step: weighted updates
        Nk = r.sum(0) + 1e-9
        mus = (r.T @ X) / Nk[:, None]
        covs = np.array([(r[:, c, None] * (X - mus[c])).T @ (X - mus[c]) / Nk[c]
                         + 1e-6 * np.eye(d) for c in range(k)])
        pis = Nk / n
    return hist


EM = em_history(X_gmm, k=3, seed=4)
RGB = np.array([mcolors.to_rgb(PALETTE[c]) for c in range(3)])


def draw_ellipse(ax, mu, cov, color):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    w, h = 2 * 2 * np.sqrt(np.maximum(vals, 1e-9))        # 2-sigma ellipse
    ax.add_patch(Ellipse(mu, w, h, angle=angle, fill=False, edgecolor=color, lw=2.5))


def show_em(iteration=0):
    pis, mus, covs, r, ll = EM[min(iteration, len(EM) - 1)]
    colors = np.clip(r @ RGB, 0, 1)                       # soft blended color per point
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(X_gmm[:, 0], X_gmm[:, 1], color=colors, s=16, alpha=0.7)
    for c in range(3):
        draw_ellipse(ax, mus[c], covs[c], PALETTE[c])
        ax.scatter(*mus[c], color=PALETTE[c], marker="X", s=180, edgecolor="white")
    ax.set_title(f"EM, итерация {iteration}/{len(EM)-1}  ·  log-правдоподобие = {ll:.0f}")
    plt.show()


if HAVE_WIDGETS:
    interact(show_em, iteration=IntSlider(min=0, max=len(EM) - 1, step=1, value=0))
else:
    show_em(len(EM) - 1)

## 5. Связь с K-Means

K-Means — это **частный предельный случай** GMM. Если зафиксировать у всех компонент
одинаковую сферическую ковариацию $\Sigma_k = \varepsilon I$ и устремить $\varepsilon \to 0$,
то ответственности из мягких $(0.55, 0.45)$ превращаются в жёсткие $(1, 0)$ — каждая точка
полностью «уходит» к ближайшему центру, а M-шаг превращается в пересчёт центра как простого
среднего. Получается ровно алгоритм Ллойда.

Отсюда практические следствия:
- GMM со `covariance_type='spherical'` ≈ «мягкий K-Means»;
- полная ковариация (`'full'`) даёт эллиптические кластеры, недоступные K-Means;
- поэтому на **вытянутых** данных, где K-Means ошибался (ноутбук 2), GMM справляется:

In [ ]:
# Anisotropic data where K-Means failed in NB2 — GMM with full covariance handles it
Xa, ya = make_blobs(n_samples=600, random_state=170)     # canonical stretched-blobs example
Xa = Xa @ np.array([[0.6, -0.6], [-0.4, 0.8]])

km = KMeans(3, n_init=10, random_state=0).fit_predict(Xa)
gm = GaussianMixture(3, covariance_type="full", n_init=5, random_state=0).fit_predict(Xa)
print("ARI  K-Means:", round(adjusted_rand_score(ya, km), 3),
      "|  GMM(full):", round(adjusted_rand_score(ya, gm), 3))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
a1.scatter(Xa[:, 0], Xa[:, 1], c=[PALETTE[i] for i in km], s=10); a1.set_title("K-Means (режет поперёк)")
a2.scatter(Xa[:, 0], Xa[:, 1], c=[PALETTE[i] for i in gm], s=10); a2.set_title("GMM full (по форме облаков)")
plt.show()

## 6. Типы ковариаций

Полная ковариация $\Sigma_k$ для каждого кластера — гибко, но дорого (много параметров, риск
переобучения на малых данных). `sklearn` предлагает компромиссы через `covariance_type`:

| `covariance_type` | что допускает | параметров на кластер |
|---|---|---|
| `'spherical'` | круг (один радиус) — «мягкий K-Means» | 1 |
| `'diag'` | оси-эллипс, выровненный по координатам | $d$ |
| `'tied'` | одна общая форма $\Sigma$ на все кластеры | общая |
| `'full'` | произвольный повёрнутый эллипс у каждого | $d(d{+}1)/2$ |

Чем «богаче» тип, тем сложнее формы он ловит, но тем больше данных нужно для надёжной оценки.

In [ ]:
# Same data, four covariance types
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, ctype in zip(axes, ["spherical", "diag", "tied", "full"]):
    lab = GaussianMixture(3, covariance_type=ctype, n_init=5, random_state=0).fit_predict(X_gmm)
    ax.scatter(X_gmm[:, 0], X_gmm[:, 1], c=[PALETTE[i] for i in lab], s=10)
    ax.set_title(f"covariance_type='{ctype}'")
plt.show()

## 7. Сколько компонент? Критерии BIC и AIC

У GMM, в отличие от K-Means, есть честное **правдоподобие**, а значит можно применять
информационные критерии, которые **штрафуют сложность** модели (число параметров $p$):

$$ \mathrm{BIC} = -2\log\hat{L} + p\log N, \qquad \mathrm{AIC} = -2\log\hat{L} + 2p. $$

Чем меньше — тем лучше. Перебираем число компонент и берём минимум: правдоподобие растёт с
числом компонент, но штраф не даёт «расти бесконечно», и появляется выраженный минимум. Это
способ **автоматически выбрать число кластеров**.

In [ ]:
ns = list(range(1, 9))
bic = [GaussianMixture(n, covariance_type="full", n_init=3, random_state=0).fit(X_gmm).bic(X_gmm) for n in ns]
aic = [GaussianMixture(n, covariance_type="full", n_init=3, random_state=0).fit(X_gmm).aic(X_gmm) for n in ns]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ns, bic, "o-", color="#2d5d8a", label="BIC")
ax.plot(ns, aic, "o-", color="#9a6b00", label="AIC")
best = ns[int(np.argmin(bic))]
ax.axvline(best, color="#9b3b46", ls="--", label=f"минимум BIC при n={best}")
ax.set_xlabel("число компонент"); ax.set_ylabel("критерий (меньше — лучше)"); ax.legend()
ax.set_title("Выбор числа компонент по BIC/AIC")
plt.show()
print("BIC выбрал компонент:", best, "(в данных их 3)")

## 8. Мягкие назначения и уверенность

Главная «фишка» GMM — метод `predict_proba`: вместо жёсткой метки он выдаёт **вероятности**
принадлежности к каждому кластеру. На **перекрывающихся** данных это особенно наглядно: точки
в зоне перекрытия получают уверенность около 0.5, и их легко подсветить.

In [ ]:
# Two overlapping clouds: where they meet, the model is honestly unsure
rng = np.random.default_rng(2)
X_ov = np.vstack([rng.multivariate_normal([0, 0],   [[1, 0], [0, 1]], 300),
                  rng.multivariate_normal([2.6, 0], [[1, 0], [0, 1]], 300)])
g_ov = GaussianMixture(2, covariance_type="full", n_init=5, random_state=0).fit(X_ov)
lab_ov = g_ov.predict(X_ov)
conf_ov = g_ov.predict_proba(X_ov).max(1)
unc = conf_ov < 0.7

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
a1.scatter(X_ov[~unc, 0], X_ov[~unc, 1], c=[PALETTE[i] for i in lab_ov[~unc]], s=14, alpha=0.6)
a1.scatter(X_ov[unc, 0], X_ov[unc, 1], facecolor="none", edgecolor="black", s=45,
           linewidth=1.2, label="неуверенные (<0.7)")
a1.legend(loc="upper right"); a1.set_title(f"Зона перекрытия: {unc.mean()*100:.0f}% неуверенных точек")
a2.hist(conf_ov, bins=30, color="#2f6f4f", alpha=0.85)
a2.set_xlabel("max вероятность"); a2.set_ylabel("число точек")
a2.set_title("Распределение уверенности")
plt.show()

### На реальных эмбеддингах дикторов

А теперь дикторы. Здесь голоса различаются настолько сильно, что модель уверена почти везде —
это хороший знак, данные «лёгкие». Смотрим качество и кросс-таблицу «диктор × кластер»:

In [ ]:
def load_speaker_embeddings(candidates=("speaker_emb.npz", "data/speaker_emb.npz")):
    # Local cache first; otherwise pull ONLY embedding/label columns from HuggingFace
    # (skips the 2.65 GB audio), then cache. One-time remote step ~1-2 min, then instant.
    import os
    for c in candidates:
        if os.path.exists(c):
            d = np.load(c, allow_pickle=True)
            return d["X"], d["y"], d["text"]
    from huggingface_hub import HfApi
    import pyarrow.parquet as pq
    import fsspec
    repo = "aiacademy-kg/audio-3-speaker-dataset"
    files = [f for f in HfApi().list_repo_files(repo, repo_type="dataset") if f.endswith(".parquet")]
    fs = fsspec.filesystem("hf")
    E, Y, T = [], [], []
    for f in files:
        pf = pq.ParquetFile(fs.open(f"datasets/{repo}/{f}"))
        d = pf.read(columns=["wavlm_embedding", "speaker_id", "text"]).to_pandas()
        E.append(np.array(d["wavlm_embedding"].tolist(), dtype=np.float32))
        Y += d["speaker_id"].tolist(); T += d["text"].tolist()
    X, y, text = np.vstack(E), np.array(Y), np.array(T, dtype=object)
    np.savez_compressed("speaker_emb.npz", X=X, y=y, text=text)
    return X, y, text


X_emb, y_emb, _ = load_speaker_embeddings()
Pe = PCA(n_components=10, random_state=0).fit_transform(StandardScaler().fit_transform(X_emb))

gmm = GaussianMixture(3, covariance_type="full", n_init=5, random_state=0).fit(Pe)
pred = gmm.predict(Pe)
conf = gmm.predict_proba(Pe).max(1)

print("ARI:", round(adjusted_rand_score(y_emb, pred), 3),
      "| NMI:", round(normalized_mutual_info_score(y_emb, pred), 3))
print("Средняя уверенность:", round(conf.mean(), 3), "— дикторы легко различимы")
print("\nДиктор × кластер:")
print(pd.crosstab(y_emb, pred))

## 9. Выводы

- **GMM** моделирует данные как **смесь гауссиан** $p(x)=\sum_k \pi_k\mathcal{N}(x\mid\mu_k,\Sigma_k)$;
  кластеры могут быть вытянутыми эллипсами.
- **Ответственности** $\gamma_{nk}$ — мягкие, вероятностные назначения; жёсткая метка — это
  $\arg\max_k\gamma_{nk}$.
- **EM** чередует **E-шаг** (пересчёт $\gamma$) и **M-шаг** (взвешенные обновления
  $\mu,\Sigma,\pi$); правдоподобие монотонно растёт к локальному максимуму.
- **K-Means — предельный случай GMM** (сферическая ковариация, жёсткие назначения).
- **`covariance_type`** управляет богатством форм; **BIC/AIC** позволяют выбрать число
  компонент автоматически.
- На эмбеддингах дикторов GMM не только разделил их, но и дал **оценку уверенности** для
  каждой записи.

**Дальше:** методы, которые сами находят число кластеров и умеют помечать **шум**, —
**DBSCAN и HDBSCAN** (кластеризация по плотности).

## Вопросы и ответы

**1) Чем GMM принципиально лучше K-Means?**
Мягкие вероятностные назначения (видно неуверенность) и эллиптические кластеры произвольной
ориентации. Плюс честное правдоподобие → можно выбирать число компонент по BIC.

**2) EM гарантирует глобальный оптимум?**
Нет, только локальный максимум правдоподобия. Поэтому инициализируют через K-Means и делают
несколько запусков (`n_init`).

**3) Почему иногда возникает ошибка про вырожденную ковариацию?**
Если компонента «схлопывается» на несколько точек, её ковариация становится сингулярной, а
правдоподобие — бесконечным. Лекарства: регуляризация (`reg_covar`), меньше компонент, проще
`covariance_type`, больше данных.

**4) Какой `covariance_type` выбрать?**
`'full'` — когда данных достаточно и формы сложные. `'diag'`/`'spherical'` — когда данных мало
или размерность большая (меньше параметров, устойчивее). Сравнивайте по BIC.

**5) GMM всегда лучше K-Means?**
Нет. На по-настоящему сферических кластерах K-Means быстрее и не хуже. GMM дороже и
чувствительнее к инициализации и вырождению.

**6) Можно ли использовать GMM для поиска аномалий?**
Да: объекты с очень низкой плотностью $p(x)$ (или низкой уверенностью) — кандидаты в выбросы.
Это частое применение помимо собственно кластеризации.